# Step 8 — Donor-balanced gene-expression trajectories along pseudotime (Python / scanpy)

Python re-implementation and extension of Step 7's §7.6 zonation-validation
figure. Given an `AnnData` object carrying

* a per-cell **pseudotime** (`obs['pseudotime']`, e.g. Monocle3 AV position),
* a **cell-type** label (`obs['Cell_type']`), and
* a **donor** label (`obs['individualID']`),

this notebook plots smooth curves of selected genes' **log1p-normalized
expression** against pseudotime, restricted to a selected cell type.

**Why "donor-balanced"?** A naive smoother (`geom_smooth` / pooled LOESS over
all cells, as in the R version) weights every *cell* equally, so a donor
contributing 8,000 nuclei dominates one contributing 300 — and if donors are
unevenly distributed along pseudotime, the resulting curve mixes real
trajectory signal with donor identity. Here every curve is instead built as:

1. fit a smooth curve **within each donor** separately, on a shared pseudotime
   grid (never extrapolating past that donor's own pseudotime range);
2. remove additive per-donor baseline shifts (the analogue of residualizing on
   `(1|individualID)`), then **average across donors with equal weight** at each
   grid point;
3. report the across-donor SEM as a ribbon, and the number of contributing
   donors, so thinly-supported stretches of the trajectory are visible rather
   than silently smoothed over.

The pooled (donor-unbalanced) curve is drawn alongside as a dashed grey line so
the size of the correction is always visible.

> Note on terminology: this is *pseudotime* (a per-cell ordinate), not
> pseudobulk. Donor balancing is what makes the curve behave like a
> per-donor-pseudobulk average within each pseudotime window — see §8.7 for an
> explicit binned-pseudobulk version of the same computation.

## 8.0 Imports

In [ ]:
import os
import warnings
from typing import Dict, Iterable, List, Optional, Sequence

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import anndata as ad
import scanpy as sc

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess as _sm_lowess
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

warnings.simplefilter("ignore", category=FutureWarning)
sc.settings.verbosity = 1
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
})
print(f"scanpy {sc.__version__} | anndata {ad.__version__} | statsmodels: {HAS_STATSMODELS}")

## 8.1 Configuration

Everything you normally need to change lives in this one cell.

In [ ]:
# ----------------------------------------------------------------- input
H5AD_PATH      = "../Results/Revision_3/Endo_cleaned.h5ad"

# ----------------------------------------------------------- obs columns
CELLTYPE_KEY   = "Cell_type"
DONOR_KEY      = "individualID"
PSEUDOTIME_KEY = "pseudotime"

# Which cell type(s) to analyse.
#   "Capillary"                  -> single cell type
#   ["Capillary", "Arterial"]    -> subset to these, and (if SPLIT_BY_CELLTYPE)
#                                   draw one curve per cell type in each panel
#   None                         -> use every cell
CELLTYPE            = ["Capillary", "Arterial",'Venous','Large_Artery']
SPLIT_BY_CELLTYPE   = False

# ------------------------------------------------------------ expression
# Where the raw counts live, if .X is not already log1p-normalized.
COUNTS_LAYER   = "counts"      # set to None to use .X / .raw as the count source
TARGET_SUM     = 1e4           # scanpy normalize_total target (CP10K)
FORCE_RENORM   = False         # True = always renormalize from counts

# ----------------------------------------------------------------- genes
GENES = [
    "DKK2", "VEGFC", "ARL15", "VCAM1",        # arterial
    "SLC7A5", "SLC16A1", "MFSD2A", "INSR",   # capillary
    "RAMP3", "TSHZ2", "IL1R1", "VWF",     # venous
    'CLDN5','TJP1','FLT1'
]

# Optional: group genes for panel ordering + colouring. Set to None to skip.
GENE_GROUPS = {
    "Arterial":       ["GJA4", "GJA5", "ALPL", "SOX17"],
    "Capillary":      ["SLC7A5", "SLC16A1", "MFSD2A", "CA4"],
    "Venous":         ["SLC38A5", "ACKR1", "NR2F2", "VWF"],
    "Mural ctrl":     ["ABCC9", "KCNJ8", "PDGFRB"],
    "NVU ctrl":       ["SLC1A1", "SLC1A2", "SLC1A3"],
}
GENE_GROUPS = None

# ------------------------------------------------------------- smoothing
SMOOTHER            = "lowess"   # "gaussian" (kernel regression) or "lowess"
N_GRID              = 120          # points on the shared pseudotime grid
BW_FRAC             = 0.08         # gaussian bandwidth, as a fraction of the pt range
LOWESS_FRAC         = 0.50         # lowess span (only used if SMOOTHER == "lowess")
MIN_CELLS_PER_DONOR = 50           # donors with fewer cells are dropped entirely
MIN_EFF_N           = 5.0          # min effective sample size behind a grid point
MIN_DONORS          = 3            # min donors contributing to a plotted grid point
PT_TRIM             = (0.0, 1.0)   # quantile trim of the pseudotime axis, e.g. (0.01, 0.99)
ALIGN_DONOR_OFFSETS = True         # remove per-donor baseline shifts before averaging (see 8.5)

# --------------------------------------------------------------- outputs
FIG_DIR    = "../Results/Revision_3/Figures"
RES_DIR    = "../Results/Revision_3"
OUT_PREFIX = "08_pseudotime_gene_curves"
SAVE       = True

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

## 8.2 Load the object and validate the required `obs` columns

In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
print(adata)

missing = [k for k in (CELLTYPE_KEY, DONOR_KEY, PSEUDOTIME_KEY) if k not in adata.obs.columns]
if missing:
    raise KeyError(
        f"missing obs columns: {missing}\n"
        f"available: {list(adata.obs.columns)}"
    )

print("\ncell types:")
print(adata.obs[CELLTYPE_KEY].value_counts())
print(f"\nn donors ({DONOR_KEY}): {adata.obs[DONOR_KEY].nunique()}")
pt = pd.to_numeric(adata.obs[PSEUDOTIME_KEY], errors="coerce")
print(f"pseudotime: range [{np.nanmin(pt):.3g}, {np.nanmax(pt):.3g}], "
      f"{int(pt.isna().sum())} NaN / {adata.n_obs} cells")


## 8.3 Normalization → log1p

Heuristic: if `.X` holds integers it is treated as raw counts and
`normalize_total(target_sum) + log1p` is applied; if it holds large
non-integers it is treated as CPM-like and only `log1p` is applied; otherwise
it is assumed to already be log1p-normalized. The decision is printed — check
it, and override with `FORCE_RENORM` / `COUNTS_LAYER` if the guess is wrong.

In [ ]:
def _peek(X, n=20000, seed=0):
    """Sample values from a (possibly sparse) matrix for the format heuristic."""
    rng = np.random.default_rng(seed)
    if sp.issparse(X):
        v = X.data
    else:
        v = np.asarray(X).ravel()
    if v.size == 0:
        return v
    if v.size > n:
        v = v[rng.choice(v.size, n, replace=False)]
    return v


def prepare_log1p(adata, counts_layer=None, target_sum=1e4, force=False):
    """Return a copy of `adata` whose .X is log1p-normalized expression."""
    a = adata.copy()

    src = None
    if force or counts_layer is not None:
        if counts_layer is not None and counts_layer in a.layers:
            a.X = a.layers[counts_layer].copy()
            src = f"layers['{counts_layer}']"
        elif force and a.raw is not None:
            a = a.raw.to_adata()[:, :].copy() if a.raw.n_vars >= a.n_vars else a
            src = ".raw"
    if src is None:
        src = ".X"

    v = _peek(a.X)
    is_int = v.size > 0 and np.allclose(v, np.round(v))
    vmax = float(v.max()) if v.size else 0.0

    if is_int:
        print(f"[norm] {src} looks like raw counts (integers, max={vmax:g}) "
              f"-> normalize_total(target_sum={target_sum:g}) + log1p")
        sc.pp.normalize_total(a, target_sum=target_sum)
        sc.pp.log1p(a)
    elif vmax > 50:
        print(f"[norm] {src} looks size-normalized but not logged (max={vmax:.1f}) -> log1p only")
        sc.pp.log1p(a)
    else:
        print(f"[norm] {src} looks already log1p-normalized (max={vmax:.2f}) -> left as is")

    return a


adata_n = prepare_log1p(adata, counts_layer=COUNTS_LAYER,
                        target_sum=TARGET_SUM, force=FORCE_RENORM)

## 8.4 Subset to the selected cell type(s) and build a tidy expression frame

In [ ]:
# ---- gene availability -------------------------------------------------
present  = [g for g in GENES if g in adata_n.var_names]
absent   = [g for g in GENES if g not in adata_n.var_names]
if absent:
    print(f"[genes] not in var_names, dropped ({len(absent)}): {absent}")
if not present:
    raise ValueError("none of the requested genes are present in the object")
print(f"[genes] using {len(present)}/{len(GENES)}")

# ---- cells -------------------------------------------------------------
mask = np.ones(adata_n.n_obs, dtype=bool)
if CELLTYPE is not None:
    wanted = [CELLTYPE] if isinstance(CELLTYPE, str) else list(CELLTYPE)
    mask &= adata_n.obs[CELLTYPE_KEY].astype(str).isin(wanted).values
    print(f"[cells] cell type filter {wanted}: {mask.sum()} cells")

pt_all = pd.to_numeric(adata_n.obs[PSEUDOTIME_KEY], errors="coerce").values
mask &= np.isfinite(pt_all)

sub = adata_n[mask, present]
print(f"[cells] after pseudotime-NaN filter: {sub.n_obs} cells, {sub.obs[DONOR_KEY].nunique()} donors")

# ---- dense expression frame -------------------------------------------
X = sub.X
X = X.toarray() if sp.issparse(X) else np.asarray(X)

df = pd.DataFrame(X, columns=list(sub.var_names), index=sub.obs_names.astype(str))
df.insert(0, "pseudotime", pd.to_numeric(sub.obs[PSEUDOTIME_KEY], errors="coerce").values)
df.insert(1, "donor",      sub.obs[DONOR_KEY].astype(str).values)
df.insert(2, "celltype",   sub.obs[CELLTYPE_KEY].astype(str).values)

# ---- pseudotime axis / trimming ---------------------------------------
lo, hi = np.quantile(df["pseudotime"], PT_TRIM)
df = df[(df["pseudotime"] >= lo) & (df["pseudotime"] <= hi)].copy()
GRID = np.linspace(lo, hi, N_GRID)
BW   = BW_FRAC * (hi - lo)
print(f"[pt] axis [{lo:.3g}, {hi:.3g}] | grid n={N_GRID} | gaussian bw={BW:.3g}")

df.head()

### Donor support along pseudotime

Donor balancing only helps where donors actually overlap. This QC panel shows
cells per donor and where each donor sits along the trajectory — a donor
occupying only one end contributes to that end of the averaged curve and is
correctly `NaN` elsewhere (no extrapolation).

In [ ]:
n_per_donor = df["donor"].value_counts().sort_values(ascending=False)
kept_donors = n_per_donor[n_per_donor >= MIN_CELLS_PER_DONOR].index.tolist()
print(f"[donors] {len(kept_donors)}/{len(n_per_donor)} donors with >= {MIN_CELLS_PER_DONOR} cells "
      f"({n_per_donor[kept_donors].sum()}/{len(df)} cells retained)")
if len(kept_donors) < MIN_DONORS:
    raise ValueError("too few donors survive MIN_CELLS_PER_DONOR to build a balanced curve")

fig, axes = plt.subplots(1, 2, figsize=(12, max(3.0, 0.16 * len(n_per_donor))),
                         gridspec_kw={"width_ratios": [1, 2]})

cols = ["tab:blue" if d in kept_donors else "lightgrey" for d in n_per_donor.index]
axes[0].barh(np.arange(len(n_per_donor)), n_per_donor.values, color=cols)
axes[0].set_yticks(np.arange(len(n_per_donor)))
axes[0].set_yticklabels(n_per_donor.index, fontsize=6)
axes[0].invert_yaxis()
axes[0].axvline(MIN_CELLS_PER_DONOR, color="crimson", lw=0.8, ls="--")
axes[0].set_xlabel("n cells"); axes[0].set_title("cells per donor", fontsize=9)

order = list(n_per_donor.index)
for i, d in enumerate(order):
    x = df.loc[df["donor"] == d, "pseudotime"].values
    if x.size == 0:
        continue
    q = np.quantile(x, [0.05, 0.25, 0.5, 0.75, 0.95])
    c = "tab:blue" if d in kept_donors else "lightgrey"
    axes[1].plot([q[0], q[4]], [i, i], color=c, lw=0.8, alpha=0.7)
    axes[1].plot([q[1], q[3]], [i, i], color=c, lw=3.0, alpha=0.8, solid_capstyle="butt")
    axes[1].plot(q[2], i, "|", color="black", ms=6, mew=1.0)
axes[1].set_yticks([]); axes[1].invert_yaxis()
axes[1].set_xlabel("pseudotime"); axes[1].set_title("pseudotime span per donor (5-95%, IQR, median)", fontsize=9)

fig.tight_layout()
# if SAVE:
#     fig.savefig(f"{FIG_DIR}/{OUT_PREFIX}_donor_QC.png")
plt.show()

## 8.5 The donor-balanced smoother

`smooth_xy` fits one curve on the shared grid. Two backends:

* **`gaussian`** — Nadaraya–Watson kernel regression,
  $\hat{f}(t)=\sum_i w_i(t)\,y_i \big/ \sum_i w_i(t)$ with
  $w_i(t)=\exp\!\big[-\tfrac12((t-t_i)/h)^2\big]$. Evaluated directly on the
  grid, so no interpolation step, and the effective sample size
  $\sum_i w_i(t)$ is available for free as a support mask.
* **`lowess`** — statsmodels LOWESS, then linear interpolation onto the grid.

Both mask grid points outside the donor's own observed pseudotime range and
grid points with effective support below `MIN_EFF_N`.

`donor_balanced_curve` runs that per donor and takes an **unweighted mean
across donors** — the balancing step — returning per-grid-point mean, SD, SEM
and donor count, plus the full donor × grid matrix for inspection.

**Donor-offset alignment (`ALIGN_DONOR_OFFSETS`).** Donors differ in overall
expression level (depth, batch, biology). Because donors also cover different
stretches of pseudotime, a plain average develops step artefacts exactly where
a high- or low-baseline donor enters or leaves the trajectory — an apparent
"change along pseudotime" that is really a change in donor composition. The fix
is the same one used in Step 7 §7.3, where expression was residualized on
`(1|individualID)` before averaging: here each donor curve is shifted by a
constant chosen so the donor curves agree on their region of overlap (a two-way
additive fit, `curve ≈ f(pseudotime) + donor_offset`, solved by a few
alternating passes), with the grand level added back so absolute log1p values
stay interpretable. Only the level is adjusted — each donor's *shape* is
untouched. Set `ALIGN_DONOR_OFFSETS = False` to see the raw version; §8.6b
plots both.

In [ ]:
def smooth_xy(x, y, grid, method="gaussian", bw=None, frac=0.5,
              min_eff_n=5.0, clip_to_range=True):
    """Smooth y over x, evaluated on `grid`. Returns (values, effective_n)."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    out  = np.full(grid.shape, np.nan)
    effn = np.zeros(grid.shape)
    if x.size == 0:
        return out, effn

    if method == "gaussian":
        h = bw if bw is not None else 0.08 * (np.ptp(grid) or 1.0)
        # (n_grid x n_cells) weights; chunked to bound memory on large donors
        chunk = max(1, int(2e7 // max(len(grid), 1)))
        num = np.zeros(len(grid)); den = np.zeros(len(grid))
        for s in range(0, x.size, chunk):
            xs, ys = x[s:s + chunk], y[s:s + chunk]
            W = np.exp(-0.5 * ((grid[:, None] - xs[None, :]) / h) ** 2)
            num += W @ ys
            den += W.sum(axis=1)
        effn = den
        with np.errstate(invalid="ignore", divide="ignore"):
            out = np.where(den > 0, num / den, np.nan)

    elif method == "lowess":
        if not HAS_STATSMODELS:
            raise ImportError("SMOOTHER='lowess' requires statsmodels")
        fit = _sm_lowess(y, x, frac=frac, it=1, return_sorted=True,
                         delta=0.01 * (np.ptp(x) or 1.0))
        out = np.interp(grid, fit[:, 0], fit[:, 1], left=np.nan, right=np.nan)
        # local density as a support proxy, on the same scale as the gaussian effn
        h = bw if bw is not None else 0.08 * (np.ptp(grid) or 1.0)
        effn = np.array([np.sum(np.abs(x - g) <= h) for g in grid], float)
    else:
        raise ValueError(f"unknown smoother: {method!r}")

    out = np.where(effn >= min_eff_n, out, np.nan)
    if clip_to_range:
        out = np.where((grid >= x.min()) & (grid <= x.max()), out, np.nan)
    return out, effn


def align_donor_offsets(per_donor, n_iter=5):
    """Remove additive per-donor level shifts (curve ~ f(pt) + donor_offset).

    Alternating fit: estimate the consensus curve, take each donor's mean
    deviation from it over the grid points that donor actually covers, subtract
    it, repeat. The mean offset is held at zero so the overall expression level
    is preserved. Shapes are never altered, only shifted.
    """
    M = per_donor.to_numpy(dtype=float, copy=True)
    if M.shape[1] < 2:
        return per_donor.copy(), pd.Series(0.0, index=per_donor.columns, name="donor_offset")

    total = np.zeros(M.shape[1])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)  # all-NaN slices
        for _ in range(n_iter):
            ref = np.nanmean(M, axis=1)                       # consensus curve
            off = np.nanmean(M - ref[:, None], axis=0)        # per-donor deviation
            off = np.where(np.isfinite(off), off, 0.0)
            off = off - off.mean()                            # keep the grand level
            if np.allclose(off, 0, atol=1e-10):
                break
            M -= off[None, :]
            total += off

    out = pd.DataFrame(M, index=per_donor.index, columns=per_donor.columns)
    return out, pd.Series(total, index=per_donor.columns, name="donor_offset")


def donor_balanced_curve(data, gene, grid, donor_col="donor", pt_col="pseudotime",
                         method="gaussian", bw=None, frac=0.5,
                         min_cells_per_donor=50, min_eff_n=5.0, min_donors=3,
                         align_offsets=True):
    """Per-donor smooth curves, then an equal-weight average across donors.

    Returns
    -------
    summary : DataFrame indexed by grid, columns [mean, sd, sem, n_donors]
    per_donor : DataFrame (grid x donor) of the individual donor curves
                (offset-aligned if `align_offsets`)
    offsets : Series of the per-donor level shifts that were removed
    """
    curves = {}
    for donor, g in data.groupby(donor_col, observed=True):
        if len(g) < min_cells_per_donor:
            continue
        vals, _ = smooth_xy(g[pt_col].values, g[gene].values, grid,
                            method=method, bw=bw, frac=frac, min_eff_n=min_eff_n)
        curves[donor] = vals

    per_donor = pd.DataFrame(curves, index=pd.Index(grid, name="pseudotime"))
    offsets = pd.Series(0.0, index=per_donor.columns, name="donor_offset")
    if per_donor.shape[1] == 0:
        empty = pd.DataFrame(index=per_donor.index,
                             columns=["mean", "sd", "sem", "n_donors"], dtype=float)
        return empty, per_donor, offsets

    if align_offsets:
        per_donor, offsets = align_donor_offsets(per_donor)

    n = per_donor.notna().sum(axis=1)
    summary = pd.DataFrame({
        "mean":     per_donor.mean(axis=1, skipna=True),      # <- donor-balanced
        "sd":       per_donor.std(axis=1, ddof=1, skipna=True),
        "n_donors": n,
    })
    summary["sem"] = summary["sd"] / np.sqrt(n.clip(lower=1))
    summary.loc[n < min_donors, ["mean", "sd", "sem"]] = np.nan
    return summary[["mean", "sd", "sem", "n_donors"]], per_donor, offsets


def pooled_curve(data, gene, grid, pt_col="pseudotime", method="gaussian",
                 bw=None, frac=0.5, min_eff_n=5.0):
    """Naive cell-weighted curve over all cells -- the donor-UNbalanced comparison."""
    vals, _ = smooth_xy(data[pt_col].values, data[gene].values, grid,
                        method=method, bw=bw, frac=frac, min_eff_n=min_eff_n)
    return pd.Series(vals, index=pd.Index(grid, name="pseudotime"))

### Compute every curve once and cache the results

In [ ]:
def compute_all(data, genes, grid, split_key=None, align_offsets=None):
    """Compute balanced + pooled curves for each gene (optionally per group)."""
    if align_offsets is None:
        align_offsets = ALIGN_DONOR_OFFSETS
    groups = {"all": data} if split_key is None else {
        k: g for k, g in data.groupby(split_key, observed=True) if len(g) > 0
    }
    balanced, pooled, per_donor, offsets = {}, {}, {}, {}
    for gname, gdata in groups.items():
        for gene in genes:
            summ, pd_mat, off = donor_balanced_curve(
                gdata, gene, grid, method=SMOOTHER, bw=BW, frac=LOWESS_FRAC,
                min_cells_per_donor=MIN_CELLS_PER_DONOR,
                min_eff_n=MIN_EFF_N, min_donors=MIN_DONORS,
                align_offsets=align_offsets)
            balanced[(gname, gene)]  = summ
            per_donor[(gname, gene)] = pd_mat
            offsets[(gname, gene)]   = off
            pooled[(gname, gene)]    = pooled_curve(
                gdata, gene, grid, method=SMOOTHER, bw=BW,
                frac=LOWESS_FRAC, min_eff_n=MIN_EFF_N)
    return balanced, pooled, per_donor, offsets, list(groups)


SPLIT_KEY = "celltype" if (SPLIT_BY_CELLTYPE and df["celltype"].nunique() > 1) else None

balanced, pooled, per_donor, offsets, group_names = compute_all(df, present, GRID, split_key=SPLIT_KEY)
print(f"computed {len(balanced)} curves over {len(group_names)} group(s): {group_names}")

# tidy export
rows = []
for (gname, gene), summ in balanced.items():
    r = summ.reset_index()
    r["gene"] = gene
    r["group"] = gname
    r["pooled_unbalanced"] = pooled[(gname, gene)].values
    rows.append(r)
curves_long = pd.concat(rows, ignore_index=True)[
    ["group", "gene", "pseudotime", "mean", "sd", "sem", "n_donors", "pooled_unbalanced"]]
if SAVE:
    curves_long.to_csv(f"{RES_DIR}/{OUT_PREFIX}_values.csv", index=False)
curves_long.head()

## 8.6 Faceted panel — one gene per panel

* thick coloured line = **donor-balanced mean**
* shaded ribbon = ±1 SEM across donors
* thin translucent lines = individual donor curves (consistency check)
* dashed grey line = pooled, cell-weighted curve (the donor-unbalanced version)

In [ ]:
def _gene_order_and_colors(genes, gene_groups):
    if not gene_groups:
        return list(genes), {g: "tab:blue" for g in genes}, {}
    order, gcol, glab = [], {}, {}
    palette = plt.get_cmap("tab10").colors
    for i, (grp, gl) in enumerate(gene_groups.items()):
        for g in gl:
            if g in genes and g not in order:
                order.append(g); gcol[g] = palette[i % len(palette)]; glab[g] = grp
    for g in genes:                       # anything not covered by GENE_GROUPS
        if g not in order:
            order.append(g); gcol[g] = "grey"; glab[g] = "Other"
    return order, gcol, glab


def plot_panels(genes, ncol=4, share_y=False, show_donors=True, show_pooled=True,
                show_sem=True, group_names=("all",), figsize_per=(2.6, 2.0),
                title=None, fname=None):
    order, gcol, glab = _gene_order_and_colors(genes, GENE_GROUPS)
    nrow = int(np.ceil(len(order) / ncol))
    fig, axes = plt.subplots(nrow, ncol, sharex=True, sharey=share_y,
                             figsize=(figsize_per[0] * ncol, figsize_per[1] * nrow))
    axes = np.atleast_1d(axes).ravel()

    multi = len(group_names) > 1
    gpal  = {gn: plt.get_cmap("Dark2").colors[i % 8] for i, gn in enumerate(group_names)}

    for ax, gene in zip(axes, order):
        for gn in group_names:
            summ = balanced[(gn, gene)]
            color = gpal[gn] if multi else gcol[gene]

            if show_donors and not multi:
                pdm = per_donor[(gn, gene)]
                for d in pdm.columns:
                    ax.plot(pdm.index, pdm[d], color=color, lw=0.5, alpha=0.20, zorder=1)
            if show_sem:
                ax.fill_between(summ.index, summ["mean"] - summ["sem"],
                                summ["mean"] + summ["sem"], color=color,
                                alpha=0.25, lw=0, zorder=2)
            ax.plot(summ.index, summ["mean"], color=color, lw=1.8, zorder=4,
                    label=gn if multi else None)
            if show_pooled:
                ax.plot(GRID, pooled[(gn, gene)].values, color="0.35", lw=1.0,
                        ls="--", alpha=0.9, zorder=3)

        ttl = gene if not glab else f"{gene}\n({glab.get(gene, '')})"
        ax.set_title(ttl, fontsize=8)
        ax.margins(x=0.02)

    for ax in axes[len(order):]:
        ax.axis("off")
    for ax in axes[max(0, len(order) - ncol):len(order)]:
        ax.set_xlabel("pseudotime")
    for i in range(0, len(order), ncol):
        axes[i].set_ylabel("log1p expression\n(donor-balanced)", fontsize=7)

    handles = [Line2D([], [], color="black", lw=1.8, label="donor-balanced mean"),
               Line2D([], [], color="0.35", lw=1.0, ls="--", label="pooled (cell-weighted)")]
    if show_donors and not multi:
        handles.append(Line2D([], [], color="black", lw=0.5, alpha=0.35, label="individual donors"))
    if multi:
        handles += [Line2D([], [], color=gpal[gn], lw=1.8, label=str(gn)) for gn in group_names]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles),
               frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.02))

    if title:
        fig.suptitle(title, fontsize=11, y=1.005)
    fig.tight_layout()
    if SAVE and fname:
    #     fig.savefig(f"{FIG_DIR}/{fname}.png")
        fig.savefig(f"{FIG_DIR}/{fname}.pdf")
    plt.show()
    return fig


ct_label = ("all cells" if CELLTYPE is None
            else CELLTYPE if isinstance(CELLTYPE, str) else ", ".join(CELLTYPE))
_ = plot_panels(
    present, ncol=5, share_y=False, group_names=group_names,
    title=(f"Donor-balanced expression along pseudotime — {ct_label}   "
           f"(n={df['donor'].nunique()} donors, {len(df):,} cells)"),
    fname=f"{OUT_PREFIX}_panels")